In [ ]:
import os
import numpy as np
from sklearn.metrics import r2_score
from sklearn.metrics.pairwise import cosine_similarity

# ----------------- 全局配置 -----------------
n_folds   = 10
embed_dir = './k_folds_model/embeddings'
pred_dir  = './k_folds_model/xgb_predictions'
GRID_SIZE = 40

# ----------------- 超参列表 -----------------
k_list       = np.arange(0.05, 1.05, 0.05)       # 0.05, 0.10, …, 0.95
a_list       = [10, 100, 500, 1000, 10000, 100000]
epsilon_list = [1e-3, 1e-4, 1e-5, 1e-6]

# ----------------- AD 辅助函数 -----------------
def compute_weights(sim_vals, a, epsilon):
    # 将 cosine 相似度从 [-1,1] 映射到 [0,1] 并 clip
    s = (sim_vals + 1.0) / 2.0
    s = np.clip(s, epsilon, 1.0)
    return np.exp(-a * (1 - s) / s)

def compute_SWD_topk(X_train, y_train, k_swd, a, epsilon):
    sim_train = np.clip(cosine_similarity(X_train), 0.0, 1.0)
    N = len(y_train)
    k_eff = max(1, min(k_swd, N-1))
    SWD = np.zeros(N)
    for t in range(N):
        sims_t = sim_train[t]
        if k_eff < N-1:
            idxs = np.argpartition(sims_t, -(k_eff+1))[-(k_eff+1):]
            idxs = idxs[idxs != t]
        else:
            idxs = np.delete(np.arange(N), t)
        if idxs.size > 0:
            sv   = sims_t[idxs]
            w    = compute_weights(sv, a, epsilon)
            diffs = np.abs(y_train[t] - y_train[idxs])
            SWD[t] = (w * sv * diffs).sum() / (w.sum() + epsilon)
    return SWD

def compute_rho_IA_topk(X_tr, y_tr, X_te, k, k_swd, a, epsilon):
    SWD    = compute_SWD_topk(X_tr, y_tr, k_swd, a, epsilon)
    sim_qt = np.clip(cosine_similarity(X_te, X_tr), 0.0, 1.0)
    M      = X_te.shape[0]
    rho_s  = np.zeros(M)
    IA     = np.zeros(M)
    for i in range(M):
        sims = sim_qt[i]
        idxk = np.argpartition(sims, -k)[-k:]
        sk   = sims[idxk]
        w    = compute_weights(sk, a, epsilon)
        rho_s[i] = w.mean()
        IA[i]    = (w * SWD[idxk]).sum() / (w.sum() + epsilon)
    return rho_s, IA


k_ratio_a=[]
a_opt_a=[]
best_R2_in_a=[]

# ----------------- 以 k 为最外层做超参搜索 -----------------
for k_ratio in k_list:
    best_R2_in = -np.inf
    best_params = None

    for a_weight in a_list:
        for eps in epsilon_list:
            # 收集所有折的 rho_s 和 IA
            all_rho, all_IA = [], []
            for fold in range(1, n_folds+1):
                X_tr = np.load(f'{embed_dir}/train_fold_{fold}.npy')[:, :768]
                y_tr = np.load(f'{embed_dir}/train_labels_fold_{fold}.npy')
                X_te = np.load(f'{embed_dir}/val_fold_{fold}.npy')[:, :768]
                k0   = max(int(k_ratio * X_tr.shape[0]), 1)
                rho_s, IA = compute_rho_IA_topk(
                    X_tr, y_tr, X_te, k0, k0, a_weight, eps
                )
                all_rho.append(rho_s)
                all_IA .append(IA)
            all_rho = np.concatenate(all_rho)
            all_IA  = np.concatenate(all_IA)

            # 构建阈值网格
            rho_cuts = np.linspace(all_rho.min(), all_rho.max(), GRID_SIZE)
            IA_cuts  = np.linspace(all_IA.min(),  all_IA.max(),  GRID_SIZE)

            # 统计平均 R²_in
            mean_r2_in = np.zeros((GRID_SIZE, GRID_SIZE))
            for fold in range(1, n_folds+1):
                X_tr = np.load(f'{embed_dir}/train_fold_{fold}.npy')[:, :768]
                y_tr = np.load(f'{embed_dir}/train_labels_fold_{fold}.npy')
                X_te = np.load(f'{embed_dir}/val_fold_{fold}.npy')[:, :768]
                y_te = np.load(f'{embed_dir}/val_labels_fold_{fold}.npy')
                y_pr= np.load(f'{pred_dir}/fold_{fold}_y_pred.npy')
                k1  = max(int(k_ratio * X_tr.shape[0]), 1)
                rho_s, IA = compute_rho_IA_topk(
                    X_tr, y_tr, X_te, k1, k1, a_weight, eps
                )
                for i, ia_th in enumerate(IA_cuts):
                    for j, rho_th in enumerate(rho_cuts):
                        in_m = (rho_s >= rho_th) & (IA <= ia_th)
                        if in_m.sum() > 1:
                            mean_r2_in[i, j] += r2_score(y_te[in_m], y_pr[in_m])
            mean_r2_in /= n_folds

            # 更新最佳 in-AD R²
            cur_best = np.nanmax(mean_r2_in)
            if cur_best > best_R2_in:
                best_R2_in  = cur_best
                best_params = (a_weight, eps)

    # 输出每轮 k 的最优结果
    a_opt, eps_opt = best_params
    print(f"k = {k_ratio:.2f}  ⇒  best a = {a_opt}, ε = {eps_opt:.0e}, max R²_in = {best_R2_in:.4f}")
    k_ratio_a.append(k_ratio)
    a_opt_a.append(a_opt)
    best_R2_in_a.append(best_R2_in)

k = 0.05  ⇒  best a = 1000, ε = 1e-03, max R²_in = 0.7748
k = 0.10  ⇒  best a = 100, ε = 1e-03, max R²_in = 0.7752
k = 0.15  ⇒  best a = 1000, ε = 1e-03, max R²_in = 0.7723
k = 0.20  ⇒  best a = 1000, ε = 1e-03, max R²_in = 0.7711


In [ ]:
import os
import numpy as np
from sklearn.metrics import r2_score
from sklearn.metrics.pairwise import cosine_similarity

# ----------------- 全局配置 -----------------
n_folds   = 10
embed_dir = './k_folds_model/embeddings'
pred_dir  = './k_folds_model/xgb_predictions'
GRID_SIZE = 40

# ----------------- 超参列表 -----------------
k_list       = np.arange(0.25, 1.05, 0.05)       # 0.05, 0.10, …, 0.95
a_list       = [10, 100, 500, 1000, 10000, 100000]
epsilon_list = [1e-3, 1e-4, 1e-5, 1e-6]

# ----------------- AD 辅助函数 -----------------
def compute_weights(sim_vals, a, epsilon):
    # 将 cosine 相似度从 [-1,1] 映射到 [0,1] 并 clip
    s = (sim_vals + 1.0) / 2.0
    s = np.clip(s, epsilon, 1.0)
    return np.exp(-a * (1 - s) / s)

def compute_SWD_topk(X_train, y_train, k_swd, a, epsilon):
    sim_train = np.clip(cosine_similarity(X_train), 0.0, 1.0)
    N = len(y_train)
    k_eff = max(1, min(k_swd, N-1))
    SWD = np.zeros(N)
    for t in range(N):
        sims_t = sim_train[t]
        if k_eff < N-1:
            idxs = np.argpartition(sims_t, -(k_eff+1))[-(k_eff+1):]
            idxs = idxs[idxs != t]
        else:
            idxs = np.delete(np.arange(N), t)
        if idxs.size > 0:
            sv   = sims_t[idxs]
            w    = compute_weights(sv, a, epsilon)
            diffs = np.abs(y_train[t] - y_train[idxs])
            SWD[t] = (w * sv * diffs).sum() / (w.sum() + epsilon)
    return SWD

def compute_rho_IA_topk(X_tr, y_tr, X_te, k, k_swd, a, epsilon):
    SWD    = compute_SWD_topk(X_tr, y_tr, k_swd, a, epsilon)
    sim_qt = np.clip(cosine_similarity(X_te, X_tr), 0.0, 1.0)
    M      = X_te.shape[0]
    rho_s  = np.zeros(M)
    IA     = np.zeros(M)
    for i in range(M):
        sims = sim_qt[i]
        idxk = np.argpartition(sims, -k)[-k:]
        sk   = sims[idxk]
        w    = compute_weights(sk, a, epsilon)
        rho_s[i] = w.mean()
        IA[i]    = (w * SWD[idxk]).sum() / (w.sum() + epsilon)
    return rho_s, IA


k_ratio_a=[]
a_opt_a=[]
best_R2_in_a=[]

# ----------------- 以 k 为最外层做超参搜索 -----------------
for k_ratio in k_list:
    best_R2_in = -np.inf
    best_params = None

    for a_weight in a_list:
        for eps in epsilon_list:
            # 收集所有折的 rho_s 和 IA
            all_rho, all_IA = [], []
            for fold in range(1, n_folds+1):
                X_tr = np.load(f'{embed_dir}/train_fold_{fold}.npy')[:, :768]
                y_tr = np.load(f'{embed_dir}/train_labels_fold_{fold}.npy')
                X_te = np.load(f'{embed_dir}/val_fold_{fold}.npy')[:, :768]
                k0   = max(int(k_ratio * X_tr.shape[0]), 1)
                rho_s, IA = compute_rho_IA_topk(
                    X_tr, y_tr, X_te, k0, k0, a_weight, eps
                )
                all_rho.append(rho_s)
                all_IA .append(IA)
            all_rho = np.concatenate(all_rho)
            all_IA  = np.concatenate(all_IA)

            # 构建阈值网格
            rho_cuts = np.linspace(all_rho.min(), all_rho.max(), GRID_SIZE)
            IA_cuts  = np.linspace(all_IA.min(),  all_IA.max(),  GRID_SIZE)

            # 统计平均 R²_in
            mean_r2_in = np.zeros((GRID_SIZE, GRID_SIZE))
            for fold in range(1, n_folds+1):
                X_tr = np.load(f'{embed_dir}/train_fold_{fold}.npy')[:, :768]
                y_tr = np.load(f'{embed_dir}/train_labels_fold_{fold}.npy')
                X_te = np.load(f'{embed_dir}/val_fold_{fold}.npy')[:, :768]
                y_te = np.load(f'{embed_dir}/val_labels_fold_{fold}.npy')
                y_pr= np.load(f'{pred_dir}/fold_{fold}_y_pred.npy')
                k1  = max(int(k_ratio * X_tr.shape[0]), 1)
                rho_s, IA = compute_rho_IA_topk(
                    X_tr, y_tr, X_te, k1, k1, a_weight, eps
                )
                for i, ia_th in enumerate(IA_cuts):
                    for j, rho_th in enumerate(rho_cuts):
                        in_m = (rho_s >= rho_th) & (IA <= ia_th)
                        if in_m.sum() > 1:
                            mean_r2_in[i, j] += r2_score(y_te[in_m], y_pr[in_m])
            mean_r2_in /= n_folds

            # 更新最佳 in-AD R²
            cur_best = np.nanmax(mean_r2_in)
            if cur_best > best_R2_in:
                best_R2_in  = cur_best
                best_params = (a_weight, eps)

    # 输出每轮 k 的最优结果
    a_opt, eps_opt = best_params
    print(f"k = {k_ratio:.2f}  ⇒  best a = {a_opt}, ε = {eps_opt:.0e}, max R²_in = {best_R2_in:.4f}")
    k_ratio_a.append(k_ratio)
    a_opt_a.append(a_opt)
    best_R2_in_a.append(best_R2_in)

k = 0.25  ⇒  best a = 1000, ε = 1e-03, max R²_in = 0.7708


In [ ]:
import os
import numpy as np
from sklearn.metrics import r2_score
from sklearn.metrics.pairwise import cosine_similarity

# ----------------- 全局配置 -----------------
n_folds   = 10
embed_dir = './k_folds_model/embeddings'
pred_dir  = './k_folds_model/xgb_predictions'
GRID_SIZE = 40

# ----------------- 超参列表 -----------------
k_list       = np.arange(0.3, 1.05, 0.05)       # 0.05, 0.10, …, 0.95
a_list       = [10, 100, 500, 1000, 10000, 100000]
epsilon_list = [1e-3, 1e-4, 1e-5, 1e-6]

# ----------------- AD 辅助函数 -----------------
def compute_weights(sim_vals, a, epsilon):
    # 将 cosine 相似度从 [-1,1] 映射到 [0,1] 并 clip
    s = (sim_vals + 1.0) / 2.0
    s = np.clip(s, epsilon, 1.0)
    return np.exp(-a * (1 - s) / s)

def compute_SWD_topk(X_train, y_train, k_swd, a, epsilon):
    sim_train = np.clip(cosine_similarity(X_train), 0.0, 1.0)
    N = len(y_train)
    k_eff = max(1, min(k_swd, N-1))
    SWD = np.zeros(N)
    for t in range(N):
        sims_t = sim_train[t]
        if k_eff < N-1:
            idxs = np.argpartition(sims_t, -(k_eff+1))[-(k_eff+1):]
            idxs = idxs[idxs != t]
        else:
            idxs = np.delete(np.arange(N), t)
        if idxs.size > 0:
            sv   = sims_t[idxs]
            w    = compute_weights(sv, a, epsilon)
            diffs = np.abs(y_train[t] - y_train[idxs])
            SWD[t] = (w * sv * diffs).sum() / (w.sum() + epsilon)
    return SWD

def compute_rho_IA_topk(X_tr, y_tr, X_te, k, k_swd, a, epsilon):
    SWD    = compute_SWD_topk(X_tr, y_tr, k_swd, a, epsilon)
    sim_qt = np.clip(cosine_similarity(X_te, X_tr), 0.0, 1.0)
    M      = X_te.shape[0]
    rho_s  = np.zeros(M)
    IA     = np.zeros(M)
    for i in range(M):
        sims = sim_qt[i]
        idxk = np.argpartition(sims, -k)[-k:]
        sk   = sims[idxk]
        w    = compute_weights(sk, a, epsilon)
        rho_s[i] = w.mean()
        IA[i]    = (w * SWD[idxk]).sum() / (w.sum() + epsilon)
    return rho_s, IA


k_ratio_a=[]
a_opt_a=[]
best_R2_in_a=[]

# ----------------- 以 k 为最外层做超参搜索 -----------------
for k_ratio in k_list:
    best_R2_in = -np.inf
    best_params = None

    for a_weight in a_list:
        for eps in epsilon_list:
            # 收集所有折的 rho_s 和 IA
            all_rho, all_IA = [], []
            for fold in range(1, n_folds+1):
                X_tr = np.load(f'{embed_dir}/train_fold_{fold}.npy')[:, :768]
                y_tr = np.load(f'{embed_dir}/train_labels_fold_{fold}.npy')
                X_te = np.load(f'{embed_dir}/val_fold_{fold}.npy')[:, :768]
                k0   = max(int(k_ratio * X_tr.shape[0]), 1)
                rho_s, IA = compute_rho_IA_topk(
                    X_tr, y_tr, X_te, k0, k0, a_weight, eps
                )
                all_rho.append(rho_s)
                all_IA .append(IA)
            all_rho = np.concatenate(all_rho)
            all_IA  = np.concatenate(all_IA)

            # 构建阈值网格
            rho_cuts = np.linspace(all_rho.min(), all_rho.max(), GRID_SIZE)
            IA_cuts  = np.linspace(all_IA.min(),  all_IA.max(),  GRID_SIZE)

            # 统计平均 R²_in
            mean_r2_in = np.zeros((GRID_SIZE, GRID_SIZE))
            for fold in range(1, n_folds+1):
                X_tr = np.load(f'{embed_dir}/train_fold_{fold}.npy')[:, :768]
                y_tr = np.load(f'{embed_dir}/train_labels_fold_{fold}.npy')
                X_te = np.load(f'{embed_dir}/val_fold_{fold}.npy')[:, :768]
                y_te = np.load(f'{embed_dir}/val_labels_fold_{fold}.npy')
                y_pr= np.load(f'{pred_dir}/fold_{fold}_y_pred.npy')
                k1  = max(int(k_ratio * X_tr.shape[0]), 1)
                rho_s, IA = compute_rho_IA_topk(
                    X_tr, y_tr, X_te, k1, k1, a_weight, eps
                )
                for i, ia_th in enumerate(IA_cuts):
                    for j, rho_th in enumerate(rho_cuts):
                        in_m = (rho_s >= rho_th) & (IA <= ia_th)
                        if in_m.sum() > 1:
                            mean_r2_in[i, j] += r2_score(y_te[in_m], y_pr[in_m])
            mean_r2_in /= n_folds

            # 更新最佳 in-AD R²
            cur_best = np.nanmax(mean_r2_in)
            if cur_best > best_R2_in:
                best_R2_in  = cur_best
                best_params = (a_weight, eps)

    # 输出每轮 k 的最优结果
    a_opt, eps_opt = best_params
    print(f"k = {k_ratio:.2f}  ⇒  best a = {a_opt}, ε = {eps_opt:.0e}, max R²_in = {best_R2_in:.4f}")
    k_ratio_a.append(k_ratio)
    a_opt_a.append(a_opt)
    best_R2_in_a.append(best_R2_in)

k = 0.30  ⇒  best a = 1000, ε = 1e-03, max R²_in = 0.7714
k = 0.35  ⇒  best a = 1000, ε = 1e-03, max R²_in = 0.7712
k = 0.40  ⇒  best a = 1000, ε = 1e-03, max R²_in = 0.7724


In [1]:
import os
import numpy as np
from sklearn.metrics import r2_score
from sklearn.metrics.pairwise import cosine_similarity

# ----------------- 全局配置 -----------------
n_folds   = 10
embed_dir = './k_folds_model/embeddings'
pred_dir  = './k_folds_model/xgb_predictions'
GRID_SIZE = 40

# ----------------- 超参列表 -----------------
k_list       = np.arange(0.45, 1.05, 0.05)       # 0.05, 0.10, …, 0.95
a_list       = [10, 100, 500, 1000, 10000, 100000]
epsilon_list = [1e-3, 1e-4, 1e-5, 1e-6]

# ----------------- AD 辅助函数 -----------------
def compute_weights(sim_vals, a, epsilon):
    # 将 cosine 相似度从 [-1,1] 映射到 [0,1] 并 clip
    s = (sim_vals + 1.0) / 2.0
    s = np.clip(s, epsilon, 1.0)
    return np.exp(-a * (1 - s) / s)

def compute_SWD_topk(X_train, y_train, k_swd, a, epsilon):
    sim_train = np.clip(cosine_similarity(X_train), 0.0, 1.0)
    N = len(y_train)
    k_eff = max(1, min(k_swd, N-1))
    SWD = np.zeros(N)
    for t in range(N):
        sims_t = sim_train[t]
        if k_eff < N-1:
            idxs = np.argpartition(sims_t, -(k_eff+1))[-(k_eff+1):]
            idxs = idxs[idxs != t]
        else:
            idxs = np.delete(np.arange(N), t)
        if idxs.size > 0:
            sv   = sims_t[idxs]
            w    = compute_weights(sv, a, epsilon)
            diffs = np.abs(y_train[t] - y_train[idxs])
            SWD[t] = (w * sv * diffs).sum() / (w.sum() + epsilon)
    return SWD

def compute_rho_IA_topk(X_tr, y_tr, X_te, k, k_swd, a, epsilon):
    SWD    = compute_SWD_topk(X_tr, y_tr, k_swd, a, epsilon)
    sim_qt = np.clip(cosine_similarity(X_te, X_tr), 0.0, 1.0)
    M      = X_te.shape[0]
    rho_s  = np.zeros(M)
    IA     = np.zeros(M)
    for i in range(M):
        sims = sim_qt[i]
        idxk = np.argpartition(sims, -k)[-k:]
        sk   = sims[idxk]
        w    = compute_weights(sk, a, epsilon)
        rho_s[i] = w.mean()
        IA[i]    = (w * SWD[idxk]).sum() / (w.sum() + epsilon)
    return rho_s, IA


k_ratio_a=[]
a_opt_a=[]
best_R2_in_a=[]

# ----------------- 以 k 为最外层做超参搜索 -----------------
for k_ratio in k_list:
    best_R2_in = -np.inf
    best_params = None

    for a_weight in a_list:
        for eps in epsilon_list:
            # 收集所有折的 rho_s 和 IA
            all_rho, all_IA = [], []
            for fold in range(1, n_folds+1):
                X_tr = np.load(f'{embed_dir}/train_fold_{fold}.npy')[:, :768]
                y_tr = np.load(f'{embed_dir}/train_labels_fold_{fold}.npy')
                X_te = np.load(f'{embed_dir}/val_fold_{fold}.npy')[:, :768]
                k0   = max(int(k_ratio * X_tr.shape[0]), 1)
                rho_s, IA = compute_rho_IA_topk(
                    X_tr, y_tr, X_te, k0, k0, a_weight, eps
                )
                all_rho.append(rho_s)
                all_IA .append(IA)
            all_rho = np.concatenate(all_rho)
            all_IA  = np.concatenate(all_IA)

            # 构建阈值网格
            rho_cuts = np.linspace(all_rho.min(), all_rho.max(), GRID_SIZE)
            IA_cuts  = np.linspace(all_IA.min(),  all_IA.max(),  GRID_SIZE)

            # 统计平均 R²_in
            mean_r2_in = np.zeros((GRID_SIZE, GRID_SIZE))
            for fold in range(1, n_folds+1):
                X_tr = np.load(f'{embed_dir}/train_fold_{fold}.npy')[:, :768]
                y_tr = np.load(f'{embed_dir}/train_labels_fold_{fold}.npy')
                X_te = np.load(f'{embed_dir}/val_fold_{fold}.npy')[:, :768]
                y_te = np.load(f'{embed_dir}/val_labels_fold_{fold}.npy')
                y_pr= np.load(f'{pred_dir}/fold_{fold}_y_pred.npy')
                k1  = max(int(k_ratio * X_tr.shape[0]), 1)
                rho_s, IA = compute_rho_IA_topk(
                    X_tr, y_tr, X_te, k1, k1, a_weight, eps
                )
                for i, ia_th in enumerate(IA_cuts):
                    for j, rho_th in enumerate(rho_cuts):
                        in_m = (rho_s >= rho_th) & (IA <= ia_th)
                        if in_m.sum() > 1:
                            mean_r2_in[i, j] += r2_score(y_te[in_m], y_pr[in_m])
            mean_r2_in /= n_folds

            # 更新最佳 in-AD R²
            cur_best = np.nanmax(mean_r2_in)
            if cur_best > best_R2_in:
                best_R2_in  = cur_best
                best_params = (a_weight, eps)

    # 输出每轮 k 的最优结果
    a_opt, eps_opt = best_params
    print(f"k = {k_ratio:.2f}  ⇒  best a = {a_opt}, ε = {eps_opt:.0e}, max R²_in = {best_R2_in:.4f}")
    k_ratio_a.append(k_ratio)
    a_opt_a.append(a_opt)
    best_R2_in_a.append(best_R2_in)

k = 0.45  ⇒  best a = 1000, ε = 1e-03, max R²_in = 0.7741
k = 0.50  ⇒  best a = 1000, ε = 1e-03, max R²_in = 0.7762
k = 0.55  ⇒  best a = 1000, ε = 1e-03, max R²_in = 0.7764
k = 0.60  ⇒  best a = 1000, ε = 1e-03, max R²_in = 0.7765
k = 0.65  ⇒  best a = 1000, ε = 1e-03, max R²_in = 0.7765
k = 0.70  ⇒  best a = 1000, ε = 1e-03, max R²_in = 0.7765
k = 0.75  ⇒  best a = 1000, ε = 1e-03, max R²_in = 0.7765
k = 0.80  ⇒  best a = 1000, ε = 1e-03, max R²_in = 0.7765
k = 0.85  ⇒  best a = 1000, ε = 1e-03, max R²_in = 0.7765
k = 0.90  ⇒  best a = 1000, ε = 1e-03, max R²_in = 0.7765
k = 0.95  ⇒  best a = 1000, ε = 1e-03, max R²_in = 0.7765
k = 1.00  ⇒  best a = 1000, ε = 1e-03, max R²_in = 0.7765


ValueError: kth(=-1210) out of bounds (24218)